# 01 — MGTAB Data Inspection

## Mục tiêu

- Kiểm tra các file raw của MGTAB
- Xác định kiểu dữ liệu và shape
- Kiểm tra encoding của edge type và label
- Kiểm tra consistency cơ bản giữa node, edge, label và feature
- Chưa thực hiện cleaning hoặc graph EDA

In [27]:
# Nhập các thư viện để đọc và kiểm tra dữ liệu.
from pathlib import Path

import torch
import numpy as np
import pandas as pd

In [47]:
# Xác định project root và đường dẫn tới dataset.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "MGTAB"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

Project root: c:\no-fraud-friend-recommendation
Dataset path: c:\no-fraud-friend-recommendation\data\raw\MGTAB
Dataset exists: True


In [48]:
# Liệt kê các file raw đang có trong thư mục dataset.
files = sorted(DATA_DIR.iterdir())

for file in files:
    print(file.name)

edge_index.pt
edge_type.pt
edge_weight.pt
features.pt
labels_bot.pt
labels_stance.pt


In [49]:
# Tính kích thước từng file để kiểm tra dữ liệu đã upload đầy đủ.
for file in files:
    size_mb = file.stat().st_size / (1024 ** 2)

    print(
        f"{file.name:25s} "
        f"{size_mb:10.2f} MB"
    )

edge_index.pt                  25.94 MB
edge_type.pt                   12.97 MB
edge_weight.pt                  6.49 MB
features.pt                    30.66 MB
labels_bot.pt                   0.08 MB
labels_stance.pt                0.08 MB


In [50]:
# Nạp toàn bộ tensor raw của MGTAB vào CPU.
edge_index = torch.load(
    DATA_DIR / "edge_index.pt",
    map_location="cpu"
)

edge_type = torch.load(
    DATA_DIR / "edge_type.pt",
    map_location="cpu"
)

edge_weight = torch.load(
    DATA_DIR / "edge_weight.pt",
    map_location="cpu"
)

features = torch.load(
    DATA_DIR / "features.pt",
    map_location="cpu"
)

labels_bot = torch.load(
    DATA_DIR / "labels_bot.pt",
    map_location="cpu"
)

labels_stance = torch.load(
    DATA_DIR / "labels_stance.pt",
    map_location="cpu"
)

In [51]:
# Kiểm tra kiểu object của từng tensor sau khi nạp.
objects = {
    "edge_index": edge_index,
    "edge_type": edge_type,
    "edge_weight": edge_weight,
    "features": features,
    "labels_bot": labels_bot,
    "labels_stance": labels_stance,
}

for name, obj in objects.items():
    print(f"{name:15s}: {type(obj)}")

edge_index     : <class 'torch.Tensor'>
edge_type      : <class 'torch.Tensor'>
edge_weight    : <class 'torch.Tensor'>
features       : <class 'torch.Tensor'>
labels_bot     : <class 'torch.Tensor'>
labels_stance  : <class 'torch.Tensor'>


In [52]:
# In shape và dtype để xác nhận schema dữ liệu.
for name, obj in objects.items():
    if isinstance(obj, torch.Tensor):
        print(
            f"{name:15s} "
            f"shape={tuple(obj.shape)}, "
            f"dtype={obj.dtype}"
        )

edge_index      shape=(2, 1700108), dtype=torch.int64
edge_type       shape=(1700108,), dtype=torch.int64
edge_weight     shape=(1700108,), dtype=torch.float32
features        shape=(10199, 788), dtype=torch.float32
labels_bot      shape=(10199,), dtype=torch.int64
labels_stance   shape=(10199,), dtype=torch.int64


In [53]:
# Hiển thị một vài giá trị đầu để kiểm tra trực quan.
print("edge_index:")
print(edge_index[:, :10])

print("\nedge_type:")
print(edge_type[:20])

print("\nedge_weight:")
print(edge_weight[:20])

print("\nfeatures:")
print(features[:3])

print("\nlabels_bot:")
print(labels_bot[:30])

print("\nlabels_stance:")
print(labels_stance[:30])

edge_index:
tensor([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [4665, 1506, 1414, 1978,  120, 4879, 3818, 8846, 5576, 8443]])

edge_type:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

edge_weight:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.])

features:
tensor([[ 1.0000,  1.0000,  0.0000,  ...,  0.9138, -0.4022,  0.4260],
        [ 1.0000,  1.0000,  0.0000,  ...,  0.9422,  0.3295, -0.0815],
        [ 1.0000,  1.0000,  0.0000,  ...,  0.7231, -0.0565,  0.3283]])

labels_bot:
tensor([0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1])

labels_stance:
tensor([1, 2, 0, 0, 2, 1, 1, 0, 0, 2, 2, 1, 1, 0, 0, 1, 0, 2, 2, 2, 0, 1, 1, 1,
        1, 1, 0, 2, 0, 1])


In [54]:
# Đếm số edge thuộc từng relation ID.
unique_edge_types, edge_counts = torch.unique(
    edge_type,
    return_counts=True
)

for relation_id, count in zip(
    unique_edge_types.tolist(),
    edge_counts.tolist()
):
    print(
        f"relation_id={relation_id}: "
        f"{count:,} edges"
    )

relation_id=0: 308,120 edges
relation_id=1: 412,575 edges
relation_id=2: 114,516 edges
relation_id=3: 223,466 edges
relation_id=4: 77,631 edges
relation_id=5: 263,800 edges
relation_id=6: 300,000 edges


In [46]:
# Tạo mapping từ relation ID sang tên quan hệ dễ đọc.
RELATION_NAMES = {
    0: "followers",
    1: "friends",
    2: "mention",
    3: "reply",
    4: "quoted",
    5: "url",
    6: "hashtag",
}

In [55]:
# In số lượng edge kèm tên relation tương ứng.
for relation_id, count in zip(
    unique_edge_types.tolist(),
    edge_counts.tolist()
):
    name = RELATION_NAMES.get(
        relation_id,
        "unknown"
    )

    print(
        f"{relation_id} - "
        f"{name:10s}: "
        f"{count:,}"
    )

0 - followers : 308,120
1 - friends   : 412,575
2 - mention   : 114,516
3 - reply     : 223,466
4 - quoted    : 77,631
5 - url       : 263,800
6 - hashtag   : 300,000


In [56]:
# Đếm số lượng human và bot trong labels_bot.
unique_labels, label_counts = torch.unique(
    labels_bot,
    return_counts=True
)

for label, count in zip(
    unique_labels.tolist(),
    label_counts.tolist()
):
    print(
        f"label={label}: "
        f"{count:,}"
    )

label=0: 7,451
label=1: 2,748


In [39]:
# Đếm số node thuộc từng stance label.
stance_unique, stance_counts = torch.unique(
    labels_stance,
    return_counts=True
)

for label, count in zip(
    stance_unique.tolist(),
    stance_counts.tolist()
):
    print(
        f"stance={label}: "
        f"{count:,}"
    )

stance=0: 3,776
stance=1: 3,637
stance=2: 2,786


In [40]:
# Kiểm tra edge_type và edge_weight có cùng độ dài với edge_index.
num_edges = edge_index.shape[1]

print(
    "Number of edges:",
    num_edges
)

print(
    "edge_type matches:",
    len(edge_type) == num_edges
)

print(
    "edge_weight matches:",
    len(edge_weight) == num_edges
)

Number of edges: 1700108
edge_type matches: True
edge_weight matches: True


In [41]:
# Kiểm tra node ID trong edge có nằm trong range hợp lệ.
num_nodes = features.shape[0]
max_node_id = edge_index.max().item()
min_node_id = edge_index.min().item()

print("Number of feature rows:", num_nodes)
print("Min node ID:", min_node_id)
print("Max node ID:", max_node_id)
print(
    "All edge node IDs valid:",
    max_node_id < num_nodes
)

Number of feature rows: 10199
Min node ID: 0
Max node ID: 10198
All edge node IDs valid: True


In [42]:
# Kiểm tra số feature rows có khớp với các loại node label.
print(
    "Number of feature rows:",
    features.shape[0]
)

print(
    "Number of bot labels:",
    labels_bot.shape[0]
)

print(
    "Number of stance labels:",
    labels_stance.shape[0]
)

Number of feature rows: 10199
Number of bot labels: 10199
Number of stance labels: 10199


In [43]:
# Kiểm tra feature có NaN hoặc Inf trước khi dùng cho model.
print("Feature shape:", features.shape)
print("Feature dtype:", features.dtype)

print(
    "Contains NaN:",
    torch.isnan(features).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(features).any().item()
)

Feature shape: torch.Size([10199, 788])
Feature dtype: torch.float32
Contains NaN: False


Contains Inf: False


In [44]:
# Tạo bảng tổng hợp schema của toàn bộ object.
summary = pd.DataFrame({
    "object": [
        "edge_index",
        "edge_type",
        "edge_weight",
        "features",
        "labels_bot",
        "labels_stance",
    ],
    "shape": [
        str(tuple(edge_index.shape)),
        str(tuple(edge_type.shape)),
        str(tuple(edge_weight.shape)),
        str(tuple(features.shape)),
        str(tuple(labels_bot.shape)),
        str(tuple(labels_stance.shape)),
    ],
    "dtype": [
        str(edge_index.dtype),
        str(edge_type.dtype),
        str(edge_weight.dtype),
        str(features.dtype),
        str(labels_bot.dtype),
        str(labels_stance.dtype),
    ],
})

summary

,object,shape,dtype
0,edge_index,"(2, 1700108)",torch.int64
1,edge_type,"(1700108,)",torch.int64
2,edge_weight,"(1700108,)",torch.float32
3,features,"(10199, 788)",torch.float32
4,labels_bot,"(10199,)",torch.int64
5,labels_stance,"(10199,)",torch.int64
